In [20]:
import csv
from pathlib import Path

In [21]:
def read_eos_conllu(path):
    """
    Reads a conllu-like file where sentences are separated by '##'.
    Returns a list of sentences, each sentence is a list of lines.
    """
    sentences = []
    current = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()
            if line == "##":
                if current:
                    sentences.append(current)
                    current = []
            elif line:
                current.append(line)

    if current:
        sentences.append(current)

    return sentences


In [22]:
def dependency_structure(sentence_lines):
    """
    Returns a list of (HEAD, DEPREL) pairs for a sentence.
    Ignores POS and all other columns.
    """
    structure = []

    for line in sentence_lines:
        fields = line.split("\t")
        if len(fields) < 8:
            continue
        head = fields[6]
        deprel = fields[7]
        structure.append((head, deprel))

    return structure


In [23]:
def sentence_text(sentence_lines):
    """
    Reconstructs sentence text from FORM column.
    """
    return " ".join(line.split("\t")[0] for line in sentence_lines)


In [27]:
def annotation_as_string(sentence_lines):
    """
    Concatenates all token lines of a sentence into a single string.
    """
    return "\n".join(sentence_lines)


In [28]:
def compare_parsers_for_category(
    file_cds,
    file_eng,
    file_stanza,
    output_csv,
    category_name
):
    sents_cds = read_eos_conllu(file_cds)
    sents_eng = read_eos_conllu(file_eng)
    sents_sta = read_eos_conllu(file_stanza)

    assert len(sents_cds) == len(sents_eng) == len(sents_sta), \
        f"Sentence count mismatch in {category_name}"

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "Sentence",
            "Roberta_CDS_biaffine",
            "Roberta_eng_biaffine",
            "Stanza_off_the_shelf"
        ])

        for sent_cds, sent_eng, sent_sta in zip(sents_cds, sents_eng, sents_sta):

            dep_cds = dependency_structure(sent_cds)
            dep_eng = dependency_structure(sent_eng)
            dep_sta = dependency_structure(sent_sta)

            # Check disagreement: at least one difference
            if not (dep_cds == dep_eng == dep_sta):
                writer.writerow([
                    sentence_text(sent_cds),
                    annotation_as_string(sent_cds),
                    annotation_as_string(sent_eng),
                    annotation_as_string(sent_sta)
                ])


In [29]:
base = Path("/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/conllu_outputs")

compare_parsers_for_category(
    base / "ambiguous_Roberta_CDS_biaffine.conllu",
    base / "ambiguous_Roberta_eng_biaffine.conllu",
    base / "ambiguous_Stanza_off_the_shelf.conllu",
    "ambiguous_discordant.csv",
    "ambiguous"
)

compare_parsers_for_category(
    base / "grammatical_Roberta_CDS_biaffine.conllu",
    base / "grammatical_Roberta_eng_biaffine.conllu",
    base / "grammatical_Stanza_off_the_shelf.conllu",
    "grammatical_discordant.csv",
    "grammatical"
)

compare_parsers_for_category(
    base / "ungrammatical_Roberta_CDS_biaffine.conllu",
    base / "ungrammatical_Roberta_eng_biaffine.conllu",
    base / "ungrammatical_Stanza_off_the_shelf.conllu",
    "ungrammatical_discordant.csv",
    "ungrammatical"
)


In [53]:
import pandas as pd

# Load your CSV
csv_path = "/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammaticality_analysis/data/manually_annotated_full.csv"
df = pd.read_csv(csv_path)

# Count the number of sentences for each grammaticality label
counts = df['is_grammatical'].value_counts().sort_index()

print("Counts of sentences by grammaticality:")
print(counts)

# Optional: explicitly show 1, 0, -1 even if some are missing
for label in [-1, 0, 1]:
    print(f"{label}: {counts.get(label, 0)}")


Counts of sentences by grammaticality:
is_grammatical
-1.0    1333
 0.0     648
 1.0    2219
Name: count, dtype: int64
-1: 1333
0: 648
1: 2219


In [ ]:
import pandas as pd 
ambiguous = pd.read_csv("/Users/frapadovani/Desktop/CHILDES-Parser/parser/ambiguous_discordant.csv")
ambiguous

,Sentence,Roberta_CDS_biaffine,Roberta_eng_biaffine,Stanza_off_the_shelf
0,lift Purdie's tail .,lift\t_\t_\t_\t_\t_\t0\troot\t_\t_\nPurdie's\t...,lift\t_\t_\t_\t_\t_\t0\troot\t_\t_\nPurdie's\t...,lift\t_\tVERB\tVB\t_\t_\t0\troot\t_\t_\nPurdie...
1,you find Thomas's face .,you\t_\t_\t_\t_\t_\t2\tnsubj\t_\t_\nfind\t_\t_...,you\t_\t_\t_\t_\t_\t2\tnsubj\t_\t_\nfind\t_\t_...,you\t_\tPRON\tPRP\t_\t_\t2\tnsubj\t_\t_\nfind\...
2,"Mummy , don't .","Mummy\t_\t_\t_\t_\t_\t3\tvocative\t_\t_\n,\t_\...","Mummy\t_\t_\t_\t_\t_\t3\tvocative\t_\t_\n,\t_\...","Mummy\t_\tNOUN\tNN\t_\t_\t3\tvocative\t_\t_\n,..."
3,choo choo .,choo\t_\t_\t_\t_\t_\t0\troot\t_\t_\nchoo\t_\t_...,choo\t_\t_\t_\t_\t_\t0\troot\t_\t_\nchoo\t_\t_...,choo\t_\tPROPN\tNNP\t_\t_\t0\troot\t_\t_\nchoo...
4,it's Mummy's house .,it's\t_\t_\t_\t_\t_\t3\tcop\t_\t_\nMummy's\t_\...,it's\t_\t_\t_\t_\t_\t3\tcop\t_\t_\nMummy's\t_\...,it\t_\tPRON\tPRP\t_\t_\t5\tnsubj\t_\t_\n's\t_\...
...,...,...,...,...
343,"a , a witch hat .","a\t_\t_\t_\t_\t_\t3\treparandum\t_\t_\n,\t_\t_...","a\t_\t_\t_\t_\t_\t5\treparandum\t_\t_\n,\t_\t_...","a\t_\tDET\tDT\t_\t_\t5\treparandum\t_\t_\n,\t_..."
344,yeah that's how witches are page .,yeah\t_\t_\t_\t_\t_\t2\tdiscourse\t_\t_\nthat'...,yeah\t_\t_\t_\t_\t_\t3\tdiscourse\t_\t_\nthat'...,yeah\t_\tINTJ\tUH\t_\t_\t4\tdiscourse\t_\t_\nt...
345,it's on the they slide on their belly .,it's\t_\t_\t_\t_\t_\t3\tnsubj\t_\t_\non\t_\t_\...,it's\t_\t_\t_\t_\t_\t3\tnsubj\t_\t_\non\t_\t_\...,it\t_\tPRON\tPRP\t_\t_\t6\tnsubj:outer\t_\t_\n...
346,one two five six seven eight nine ten for elev...,one\t_\t_\t_\t_\t_\t0\troot\t_\t_\ntwo\t_\t_\t...,one\t_\t_\t_\t_\t_\t0\troot\t_\t_\ntwo\t_\t_\t...,one\t_\tNUM\tCD\t_\t_\t6\tnummod\t_\t_\ntwo\t_...


In [43]:
ambiguous_subset = ambiguous[ambiguous["Roberta_CDS_biaffine"] != ambiguous["Roberta_eng_biaffine"]]
ambiguous_subset

,Sentence,Roberta_CDS_biaffine,Roberta_eng_biaffine,Stanza_off_the_shelf
2,"Mummy , don't .","Mummy\t_\t_\t_\t_\t_\t3\tvocative\t_\t_\n,\t_\...","Mummy\t_\t_\t_\t_\t_\t3\tvocative\t_\t_\n,\t_\...","Mummy\t_\tNOUN\tNN\t_\t_\t3\tvocative\t_\t_\n,..."
3,choo choo .,choo\t_\t_\t_\t_\t_\t0\troot\t_\t_\nchoo\t_\t_...,choo\t_\t_\t_\t_\t_\t0\troot\t_\t_\nchoo\t_\t_...,choo\t_\tPROPN\tNNP\t_\t_\t0\troot\t_\t_\nchoo...
5,wheels on the bus go red red red .,wheels\t_\t_\t_\t_\t_\t5\tnsubj\t_\t_\non\t_\t...,wheels\t_\t_\t_\t_\t_\t5\tnsubj\t_\t_\non\t_\t...,wheels\t_\tNOUN\tNNS\t_\t_\t5\tnsubj\t_\t_\non...
6,red red red .,red\t_\t_\t_\t_\t_\t0\troot\t_\t_\nred\t_\t_\t...,red\t_\t_\t_\t_\t_\t3\tamod\t_\t_\nred\t_\t_\t...,red\t_\tADJ\tJJ\t_\t_\t3\tamod\t_\t_\nred\t_\t...
7,red red red red red .,red\t_\t_\t_\t_\t_\t0\troot\t_\t_\nred\t_\t_\t...,red\t_\t_\t_\t_\t_\t0\troot\t_\t_\nred\t_\t_\t...,red\t_\tADJ\tJJ\t_\t_\t5\tamod\t_\t_\nred\t_\t...
...,...,...,...,...
342,because it - she's flying up and she's green .,because\t_\t_\t_\t_\t_\t5\tmark\t_\t_\nit\t_\t...,because\t_\t_\t_\t_\t_\t5\tmark\t_\t_\nit\t_\t...,because\t_\tSCONJ\tIN\t_\t_\t6\tmark\t_\t_\nit...
343,"a , a witch hat .","a\t_\t_\t_\t_\t_\t3\treparandum\t_\t_\n,\t_\t_...","a\t_\t_\t_\t_\t_\t5\treparandum\t_\t_\n,\t_\t_...","a\t_\tDET\tDT\t_\t_\t5\treparandum\t_\t_\n,\t_..."
344,yeah that's how witches are page .,yeah\t_\t_\t_\t_\t_\t2\tdiscourse\t_\t_\nthat'...,yeah\t_\t_\t_\t_\t_\t3\tdiscourse\t_\t_\nthat'...,yeah\t_\tINTJ\tUH\t_\t_\t4\tdiscourse\t_\t_\nt...
345,it's on the they slide on their belly .,it's\t_\t_\t_\t_\t_\t3\tnsubj\t_\t_\non\t_\t_\...,it's\t_\t_\t_\t_\t_\t3\tnsubj\t_\t_\non\t_\t_\...,it\t_\tPRON\tPRP\t_\t_\t6\tnsubj:outer\t_\t_\n...


In [33]:
grammatical = pd.read_csv("/Users/frapadovani/Desktop/CHILDES-Parser/parser/grammatical_discordant.csv")
grammatical

,Sentence,Roberta_CDS_biaffine,Roberta_eng_biaffine,Stanza_off_the_shelf
0,oh yes .,oh\t_\t_\t_\t_\t_\t0\troot\t_\t_\nyes\t_\t_\t_...,oh\t_\t_\t_\t_\t_\t0\troot\t_\t_\nyes\t_\t_\t_...,oh\t_\tINTJ\tUH\t_\t_\t2\tdiscourse\t_\t_\nyes...
1,"Mummy , you say help .","Mummy\t_\t_\t_\t_\t_\t4\tvocative\t_\t_\n,\t_\...","Mummy\t_\t_\t_\t_\t_\t4\tvocative\t_\t_\n,\t_\...","Mummy\t_\tNOUN\tNN\t_\t_\t4\tvocative\t_\t_\n,..."
2,Wend Wendy's hat .,Wend\t_\t_\t_\t_\t_\t3\treparandum\t_\t_\nWend...,Wend\t_\t_\t_\t_\t_\t0\troot\t_\t_\nWendy's\t_...,Wend\t_\tVERB\tVB\t_\t_\t0\troot\t_\t_\nWendy\...
3,that's Wendy's hat there .,that's\t_\t_\t_\t_\t_\t3\tcop\t_\t_\nWendy's\t...,that's\t_\t_\t_\t_\t_\t3\tcop\t_\t_\nWendy's\t...,that\t_\tPRON\tDT\t_\t_\t5\tnsubj\t_\t_\n's\t_...
4,I'm not a fireman .,I'm\t_\t_\t_\t_\t_\t4\tcop\t_\t_\nnot\t_\t_\t_...,I'm\t_\t_\t_\t_\t_\t4\tcop\t_\t_\nnot\t_\t_\t_...,I\t_\tPRON\tPRP\t_\t_\t5\tnsubj\t_\t_\n'm\t_\t...
...,...,...,...,...
1276,because it's big .,because\t_\t_\t_\t_\t_\t3\tmark\t_\t_\nit's\t_...,because\t_\t_\t_\t_\t_\t3\tmark\t_\t_\nit's\t_...,because\t_\tSCONJ\tIN\t_\t_\t4\tmark\t_\t_\nit...
1277,"because , I just like it .","because\t_\t_\t_\t_\t_\t5\tmark\t_\t_\n,\t_\t_...","because\t_\t_\t_\t_\t_\t5\tmark\t_\t_\n,\t_\t_...","because\t_\tSCONJ\tIN\t_\t_\t5\tmark\t_\t_\n,\..."
1278,with the with the corn .,with\t_\t_\t_\t_\t_\t2\tcase\t_\t_\nthe\t_\t_\...,with\t_\t_\t_\t_\t_\t5\treparandum\t_\t_\nthe\...,with\t_\tADP\tIN\t_\t_\t2\tcase\t_\t_\nthe\t_\...
1279,then we put water in for for the big duck to s...,then\t_\t_\t_\t_\t_\t3\tadvmod\t_\t_\nwe\t_\t_...,then\t_\t_\t_\t_\t_\t3\tadvmod\t_\t_\nwe\t_\t_...,then\t_\tADV\tRB\t_\t_\t3\tadvmod\t_\t_\nwe\t_...


In [44]:
grammatical_subset = grammatical[grammatical["Roberta_CDS_biaffine"] != grammatical["Roberta_eng_biaffine"]]
grammatical_subset

,Sentence,Roberta_CDS_biaffine,Roberta_eng_biaffine,Stanza_off_the_shelf
0,oh yes .,oh\t_\t_\t_\t_\t_\t0\troot\t_\t_\nyes\t_\t_\t_...,oh\t_\t_\t_\t_\t_\t0\troot\t_\t_\nyes\t_\t_\t_...,oh\t_\tINTJ\tUH\t_\t_\t2\tdiscourse\t_\t_\nyes...
1,"Mummy , you say help .","Mummy\t_\t_\t_\t_\t_\t4\tvocative\t_\t_\n,\t_\...","Mummy\t_\t_\t_\t_\t_\t4\tvocative\t_\t_\n,\t_\...","Mummy\t_\tNOUN\tNN\t_\t_\t4\tvocative\t_\t_\n,..."
2,Wend Wendy's hat .,Wend\t_\t_\t_\t_\t_\t3\treparandum\t_\t_\nWend...,Wend\t_\t_\t_\t_\t_\t0\troot\t_\t_\nWendy's\t_...,Wend\t_\tVERB\tVB\t_\t_\t0\troot\t_\t_\nWendy\...
5,uh no thanks .,uh\t_\t_\t_\t_\t_\t3\tdiscourse\t_\t_\nno\t_\t...,uh\t_\t_\t_\t_\t_\t3\tdiscourse\t_\t_\nno\t_\t...,uh\t_\tINTJ\tUH\t_\t_\t3\tdiscourse\t_\t_\nno\...
7,oh no .,oh\t_\t_\t_\t_\t_\t0\troot\t_\t_\nno\t_\t_\t_\...,oh\t_\t_\t_\t_\t_\t2\tdiscourse\t_\t_\nno\t_\t...,oh\t_\tINTJ\tUH\t_\t_\t2\tdiscourse\t_\t_\nno\...
...,...,...,...,...
1276,because it's big .,because\t_\t_\t_\t_\t_\t3\tmark\t_\t_\nit's\t_...,because\t_\t_\t_\t_\t_\t3\tmark\t_\t_\nit's\t_...,because\t_\tSCONJ\tIN\t_\t_\t4\tmark\t_\t_\nit...
1277,"because , I just like it .","because\t_\t_\t_\t_\t_\t5\tmark\t_\t_\n,\t_\t_...","because\t_\t_\t_\t_\t_\t5\tmark\t_\t_\n,\t_\t_...","because\t_\tSCONJ\tIN\t_\t_\t5\tmark\t_\t_\n,\..."
1278,with the with the corn .,with\t_\t_\t_\t_\t_\t2\tcase\t_\t_\nthe\t_\t_\...,with\t_\t_\t_\t_\t_\t5\treparandum\t_\t_\nthe\...,with\t_\tADP\tIN\t_\t_\t2\tcase\t_\t_\nthe\t_\...
1279,then we put water in for for the big duck to s...,then\t_\t_\t_\t_\t_\t3\tadvmod\t_\t_\nwe\t_\t_...,then\t_\t_\t_\t_\t_\t3\tadvmod\t_\t_\nwe\t_\t_...,then\t_\tADV\tRB\t_\t_\t3\tadvmod\t_\t_\nwe\t_...


In [46]:
ungrammatical = pd.read_csv("/Users/frapadovani/Desktop/CHILDES-Parser/parser/ungrammatical_discordant.csv")
ungrammatical

,Sentence,Roberta_CDS_biaffine,Roberta_eng_biaffine,Stanza_off_the_shelf
0,I think stick on here .,I\t_\t_\t_\t_\t_\t2\tnsubj\t_\t_\nthink\t_\t_\...,I\t_\t_\t_\t_\t_\t2\tnsubj\t_\t_\nthink\t_\t_\...,I\t_\tPRON\tPRP\t_\t_\t2\tnsubj\t_\t_\nthink\t...
1,wanna be Fireman Sam .,wanna\t_\t_\t_\t_\t_\t0\troot\t_\t_\nbe\t_\t_\...,wanna\t_\t_\t_\t_\t_\t0\troot\t_\t_\nbe\t_\t_\...,wan\t_\tVERB\tVBP\t_\t_\t0\troot\t_\t_\nna\t_\...
2,I Fireman Sam .,I\t_\t_\t_\t_\t_\t2\tnsubj\t_\t_\nFireman\t_\t...,I\t_\t_\t_\t_\t_\t3\tdep\t_\t_\nFireman\t_\t_\...,I\t_\tPRON\tPRP\t_\t_\t2\tnsubj\t_\t_\nFireman...
3,wanna talk now .,wanna\t_\t_\t_\t_\t_\t0\troot\t_\t_\ntalk\t_\t...,wanna\t_\t_\t_\t_\t_\t0\troot\t_\t_\ntalk\t_\t...,wan\t_\tVERB\tVBP\t_\t_\t0\troot\t_\t_\nna\t_\...
4,"Purdie , that Fireman Sam .","Purdie\t_\t_\t_\t_\t_\t4\tvocative\t_\t_\n,\t_...","Purdie\t_\t_\t_\t_\t_\t5\tvocative\t_\t_\n,\t_...","Purdie\t_\tPROPN\tNNP\t_\t_\t0\troot\t_\t_\n,\..."
...,...,...,...,...
700,because that .,because\t_\t_\t_\t_\t_\t2\tmark\t_\t_\nthat\t_...,because\t_\t_\t_\t_\t_\t2\tcase\t_\t_\nthat\t_...,because\t_\tADP\tIN\t_\t_\t2\tcase\t_\t_\nthat...
701,"mommy but , Firstname really seen real leprech...",mommy\t_\t_\t_\t_\t_\t6\tvocative\t_\t_\nbut\t...,mommy\t_\t_\t_\t_\t_\t0\troot\t_\t_\nbut\t_\t_...,mommy\t_\tINTJ\tUH\t_\t_\t6\tdiscourse\t_\t_\n...
702,yeah but .,yeah\t_\t_\t_\t_\t_\t2\tdiscourse\t_\t_\nbut\t...,yeah\t_\t_\t_\t_\t_\t0\troot\t_\t_\nbut\t_\t_\...,yeah\t_\tINTJ\tUH\t_\t_\t0\troot\t_\t_\nbut\t_...
703,he swim he drinked it all up .,he\t_\t_\t_\t_\t_\t2\tnsubj\t_\t_\nswim\t_\t_\...,he\t_\t_\t_\t_\t_\t2\tnsubj\t_\t_\nswim\t_\t_\...,he\t_\tPRON\tPRP\t_\t_\t2\tnsubj\t_\t_\nswim\t...


In [55]:
print(ungrammatical.iloc[1][1])
print()
print(ungrammatical.iloc[1][2])
print()
print(ungrammatical.iloc[1][3])

wanna	_	_	_	_	_	0	root	_	_
be	_	_	_	_	_	3	cop	_	_
Fireman	_	_	_	_	_	1	xcomp	_	_
Sam	_	_	_	_	_	3	flat	_	_
.	_	_	_	_	_	1	punct	_	_

wanna	_	_	_	_	_	0	root	_	_
be	_	_	_	_	_	4	cop	_	_
Fireman	_	_	_	_	_	4	nmod:desc	_	_
Sam	_	_	_	_	_	1	xcomp	_	_
.	_	_	_	_	_	1	punct	_	_

wan	_	VERB	VBP	_	_	0	root	_	_
na	_	PART	TO	_	_	5	mark	_	_
be	_	AUX	VB	_	_	5	cop	_	_
Fireman	_	PROPN	NNP	_	_	1	xcomp	_	_
Sam	_	PROPN	NNP	_	_	4	flat	_	_
.	_	PUNCT	.	_	_	1	punct	_	_


/var/folders/hf/t5b8jlfd4lz04bzl35ccq5mm0000gn/T/ipykernel_92075/2226968966.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(ungrammatical.iloc[1][1])
/var/folders/hf/t5b8jlfd4lz04bzl35ccq5mm0000gn/T/ipykernel_92075/2226968966.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(ungrammatical.iloc[1][2])
/var/folders/hf/t5b8jlfd4lz04bzl35ccq5mm0000gn/T/ipykernel_92075/2226968966.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.i

In [45]:
ungrammatical_subset = ungrammatical[ungrammatical["Roberta_CDS_biaffine"] != ungrammatical["Roberta_eng_biaffine"]]
ungrammatical_subset

,Sentence,Roberta_CDS_biaffine,Roberta_eng_biaffine,Stanza_off_the_shelf
1,wanna be Fireman Sam .,wanna\t_\t_\t_\t_\t_\t0\troot\t_\t_\nbe\t_\t_\...,wanna\t_\t_\t_\t_\t_\t0\troot\t_\t_\nbe\t_\t_\...,wan\t_\tVERB\tVBP\t_\t_\t0\troot\t_\t_\nna\t_\...
2,I Fireman Sam .,I\t_\t_\t_\t_\t_\t2\tnsubj\t_\t_\nFireman\t_\t...,I\t_\t_\t_\t_\t_\t3\tdep\t_\t_\nFireman\t_\t_\...,I\t_\tPRON\tPRP\t_\t_\t2\tnsubj\t_\t_\nFireman...
4,"Purdie , that Fireman Sam .","Purdie\t_\t_\t_\t_\t_\t4\tvocative\t_\t_\n,\t_...","Purdie\t_\t_\t_\t_\t_\t5\tvocative\t_\t_\n,\t_...","Purdie\t_\tPROPN\tNNP\t_\t_\t0\troot\t_\t_\n,\..."
6,looking the kitchen for a Wendy's hat .,looking\t_\t_\t_\t_\t_\t0\troot\t_\t_\nthe\t_\...,looking\t_\t_\t_\t_\t_\t0\troot\t_\t_\nthe\t_\...,looking\t_\tVERB\tVBG\t_\t_\t0\troot\t_\t_\nth...
7,I not called Aunty Mabel .,I\t_\t_\t_\t_\t_\t3\tnsubj:pass\t_\t_\nnot\t_\...,I\t_\t_\t_\t_\t_\t3\tnsubj:pass\t_\t_\nnot\t_\...,I\t_\tPRON\tPRP\t_\t_\t3\tnsubj\t_\t_\nnot\t_\...
...,...,...,...,...
699,when when can I when ca-what what Grandma ?,when\t_\t_\t_\t_\t_\t2\treparandum\t_\t_\nwhen...,when\t_\t_\t_\t_\t_\t2\tadvmod\t_\t_\nwhen\t_\...,when\t_\tADV\tWRB\t_\t_\t6\tadvmod\t_\t_\nwhen...
700,because that .,because\t_\t_\t_\t_\t_\t2\tmark\t_\t_\nthat\t_...,because\t_\t_\t_\t_\t_\t2\tcase\t_\t_\nthat\t_...,because\t_\tADP\tIN\t_\t_\t2\tcase\t_\t_\nthat...
701,"mommy but , Firstname really seen real leprech...",mommy\t_\t_\t_\t_\t_\t6\tvocative\t_\t_\nbut\t...,mommy\t_\t_\t_\t_\t_\t0\troot\t_\t_\nbut\t_\t_...,mommy\t_\tINTJ\tUH\t_\t_\t6\tdiscourse\t_\t_\n...
702,yeah but .,yeah\t_\t_\t_\t_\t_\t2\tdiscourse\t_\t_\nbut\t...,yeah\t_\t_\t_\t_\t_\t0\troot\t_\t_\nbut\t_\t_\...,yeah\t_\tINTJ\tUH\t_\t_\t0\troot\t_\t_\nbut\t_...


In [51]:
print(ungrammatical_subset.iloc[1][1])
print()
print(ungrammatical_subset.iloc[1][2])
print()
print(ungrammatical_subset.iloc[1][3])

I	_	_	_	_	_	2	nsubj	_	_
Fireman	_	_	_	_	_	0	root	_	_
Sam	_	_	_	_	_	2	flat	_	_
.	_	_	_	_	_	2	punct	_	_

I	_	_	_	_	_	3	dep	_	_
Fireman	_	_	_	_	_	3	compound	_	_
Sam	_	_	_	_	_	0	root	_	_
.	_	_	_	_	_	3	punct	_	_

I	_	PRON	PRP	_	_	2	nsubj	_	_
Fireman	_	PROPN	NNP	_	_	0	root	_	_
Sam	_	PROPN	NNP	_	_	2	flat	_	_
.	_	PUNCT	.	_	_	2	punct	_	_


/var/folders/hf/t5b8jlfd4lz04bzl35ccq5mm0000gn/T/ipykernel_92075/4043848949.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(ungrammatical_subset.iloc[1][1])
/var/folders/hf/t5b8jlfd4lz04bzl35ccq5mm0000gn/T/ipykernel_92075/4043848949.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(ungrammatical_subset.iloc[1][2])
/var/folders/hf/t5b8jlfd4lz04bzl35ccq5mm0000gn/T/ipykernel_92075/4043848949.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by positi